# Benchpress-Style Workouts — SF vs. Qiskit

Provider-agnostic benchmarking using the Superfermion benchmarks module.
This notebook runs IBM Benchpress-inspired workouts across multiple SDK strategies.

**Workouts covered:**
- **Construction** (8 tests): Circuit building, parameter binding, QASM import
- **Manipulation** (4 tests): Pauli twirling, gate decomposition, basis translation
- **Transpilation** (9 tests): Full compilation to hardware topology

**Methodology:**
- Each workout runs 3 rounds (configurable)
- Mean wall-clock time reported
- Quality metrics: 2Q gate count, circuit depth after transpilation

In [1]:
# -- Bootstrap: pin the fixed LOCAL build (added for the 2026-09-24 linearfixes refresh) --
import sys as _sys
import hashlib as _hashlib
from pathlib import Path as _Path
REPO = _Path(r"C:\Users\ASUS\OneDrive\Desktop\sfdocs\superfermion")
assert (REPO / 'superfermion' / '__init__.py').exists(), f'repo not found: {REPO}'
if str(REPO) not in _sys.path:
    _sys.path.insert(0, str(REPO))
import superfermion as _sf
import superfermion._sf_core as _sfcore
_core = _Path(_sfcore.__file__)
print('=' * 78)
print('ENGINE PROVENANCE - fixed local build pinned for this run')
print('=' * 78)
print(f'  engine : {_core.name}  md5 {_hashlib.md5(_core.read_bytes()).hexdigest().upper()}')
print(f'  sf     : {_sf.__version__}  from {_Path(_sf.__file__).parent}')
print()


ENGINE PROVENANCE - fixed local build pinned for this run
  engine : _sf_core.cp313-win_amd64.pyd  md5 17B755C032D01D0C9711EA2B75751506
  sf     : 0.1.12  from C:\Users\ASUS\OneDrive\Desktop\sfdocs\superfermion\superfermion



In [2]:
from superfermion.benchmarks import BenchmarkRunner, TopologyFactory, list_strategies
from superfermion.benchmarks.strategies import SuperfermionStrategy

print(f"Available strategies: {list_strategies()}")

Available strategies: ['cirq', 'qiskit', 'superfermion']


## Setup

Configure the target hardware topology and SDK strategies.

In [3]:
# Target backend — heavy-hex topology (127Q) with ECR basis
backend = TopologyFactory.create(
    "heavy_hex",
    n_qubits=127,
    basis_gates=["rz", "sx", "x", "ecr"],
)
print(f"Backend: {backend.n_qubits}Q, {len(backend.coupling_map)} edges")
print(f"Basis: {backend.basis_gates}")

# SDK strategies
strategies = [SuperfermionStrategy()]

# Uncomment to include Qiskit (requires qiskit installed):
# from superfermion.benchmarks.strategies import QiskitStrategy
# strategies.append(QiskitStrategy())

print(f"Strategies: {[s.name for s in strategies]}")

Backend: 127Q, 174 edges
Basis: ['rz', 'sx', 'x', 'ecr']
Strategies: ['superfermion']


## 1. Construction Workouts

Measure circuit building speed: QV, DTC, Clifford, MCX, SU2, QASM import.

In [4]:
runner = BenchmarkRunner()

construction_report = runner.run(
    strategies=strategies,
    rounds=3,
    category="construction",
)
print()
print(construction_report.summary_table())

  OK  [1/8] bench_qv_build (superfermion): 0.1467s


  OK  [2/8] bench_dtc_build (superfermion): 0.1989s


  OK  [3/8] bench_clifford_build (superfermion): 0.4874s
  OK  [4/8] bench_multi_control_build (superfermion): 0.0001s
  OK  [5/8] bench_su2_build (superfermion): 0.0030s
  OK  [6/8] bench_su2_bind (superfermion): 0.0021s


  OK  [7/8] bench_qasm2_import (superfermion): 0.0584s
  OK  [8/8] bench_qasm2_bigint (superfermion): 0.0003s

Test                                     SDK                 Time (s)   Rounds
------------------------------------------------------------------------------
bench_qv_build                           superfermion          0.1467        3
bench_dtc_build                          superfermion          0.1989        3
bench_clifford_build                     superfermion          0.4874        3
bench_multi_control_build                superfermion          0.0001        3
bench_su2_build                          superfermion          0.0030        3
bench_su2_bind                           superfermion          0.0021        3
bench_qasm2_import                       superfermion          0.0584        3
bench_qasm2_bigint                       superfermion          0.0003        3


## 2. Manipulation Workouts

Test circuit transformation: Pauli twirling, decomposition, basis change.

In [5]:
manipulation_report = runner.run(
    strategies=strategies,
    rounds=3,
    category="manipulation",
)
print()
print(manipulation_report.summary_table())

  OK  [1/4] bench_pauli_twirl (superfermion): 0.0004s
  OK  [2/4] bench_multi_control_decompose (superfermion): 0.0037s


  OK  [3/4] bench_basis_change (superfermion): 0.0686s
  OK  [4/4] bench_clifford_decompose (superfermion): 0.0109s

Test                                     SDK                 Time (s)   Rounds
------------------------------------------------------------------------------
bench_pauli_twirl                        superfermion          0.0004        3
bench_multi_control_decompose            superfermion          0.0037        3
bench_basis_change                       superfermion          0.0686        3
bench_clifford_decompose                 superfermion          0.0109        3


## 3. Transpilation Workouts

End-to-end compilation to hardware topology. Uses the heavy-hex 127Q backend.

**Note:** These can take 10-60s+ per test on large circuits. Use `all_to_all` for faster runs.

In [6]:
# For quick runs, use all-to-all (no routing).
# For realistic benchmarking, use heavy_hex.
transpile_backend = TopologyFactory.create(
    "all_to_all",
    n_qubits=100,
    basis_gates=["rz", "sx", "x", "cx"],
)

transpilation_report = runner.run(
    strategies=strategies,
    rounds=1,
    category="transpilation",
    backend=transpile_backend,
)
print()
print(transpilation_report.summary_table())

  OK  [1/9] bench_qft_transpile (superfermion): 0.7004s


  OK  [2/9] bench_qv_transpile (superfermion): 2.5926s
  OK  [3/9] bench_su2_transpile (superfermion): 0.1637s
  OK  [4/9] bench_bv_transpile (superfermion): 0.0128s
  OK  [5/9] bench_heisenberg_transpile (superfermion): 0.1416s
  OK  [6/9] bench_qaoa_transpile (superfermion): 0.0494s
  OK  [7/9] bench_simplification_transpile (superfermion): 0.0083s


  OK  [8/9] bench_clifford_transpile (superfermion): 10.9256s
  OK  [9/9] bench_circsu2_89_transpile (superfermion): 0.1256s

Test                                     SDK                 Time (s)   Rounds
------------------------------------------------------------------------------
bench_qft_transpile                      superfermion          0.7004        1
bench_qv_transpile                       superfermion          2.5926        1
bench_su2_transpile                      superfermion          0.1637        1
bench_bv_transpile                       superfermion          0.0128        1
bench_heisenberg_transpile               superfermion          0.1416        1
bench_qaoa_transpile                     superfermion          0.0494        1
bench_simplification_transpile           superfermion          0.0083        1
bench_clifford_transpile                 superfermion         10.9256        1
bench_circsu2_89_transpile               superfermion          0.1256        1


## 4. Visualization & Export

In [7]:
# Merge all reports for a combined view
from superfermion.benchmarks.runner import RunnerReport

full_report = RunnerReport()
for r in construction_report.results:
    full_report.add(r)
for r in manipulation_report.results:
    full_report.add(r)
for r in transpilation_report.results:
    full_report.add(r)

print(f"Total benchmarks: {len(full_report.results)}")

Total benchmarks: 21


In [8]:
# Generate bar chart
full_report.plot(title="Benchpress Workouts — Superfermion")

<Figure size 2520x600 with 1 Axes>

In [9]:
# Export to pytest-benchmark-compatible JSON
json_path = "../benchmark_results.json"
full_report.to_json(json_path)
print(f"Results exported to {json_path}")

Results exported to ../benchmark_results.json


In [10]:
# Speedup table (useful when multiple strategies are compared)
if len(strategies) > 1:
    print(full_report.speedup_table())
else:
    print("Single strategy — speedup comparison requires 2+ SDKs.")
    print("Uncomment QiskitStrategy in setup cell to compare.")

Single strategy — speedup comparison requires 2+ SDKs.
Uncomment QiskitStrategy in setup cell to compare.


In [11]:
# Quality metrics
from superfermion.benchmarks.report import generate_quality_table
print(generate_quality_table(full_report))

Test                                     SDK            2Q Gates    Depth  Total Gates
--------------------------------------------------------------------------------------
bench_qv_build                           superfermion                            35000
bench_dtc_build                          superfermion                            29900
bench_clifford_build                     superfermion                            27173
bench_multi_control_build                superfermion                               29
bench_su2_build                          superfermion                             1400
bench_su2_bind                           superfermion                                 
bench_qasm2_import                       superfermion                            15000
bench_qasm2_bigint                       superfermion                                 
bench_pauli_twirl                        superfermion                             2990
bench_multi_control_decompose            su